In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

df = pd.read_csv('../data/ran_kpi_data.csv')
kpis = ['rsrp', 'sinr', 'prb_utilization', 'throughput_mbps', 'packet_loss', 'latency_ms']
X = df[kpis].values
y = df['is_anomaly'].values

scaler = joblib.load('../models/scaler.pkl')
X_scaled = scaler.transform(X)
X_normal = X_scaled[y == 0]

print("Training on normal samples:", X_normal.shape)

In [ ]:
autoencoder = MLPRegressor(
    hidden_layer_sizes=(16, 8, 4, 8, 16),
    activation='relu',
    max_iter=200,
    random_state=42,
    verbose=False
)
autoencoder.fit(X_normal, X_normal)
print("Training done")

In [ ]:
X_reconstructed = autoencoder.predict(X_scaled)
reconstruction_errors = np.mean(np.power(X_scaled - X_reconstructed, 2), axis=1)

normal_errors = reconstruction_errors[y == 0]
threshold = np.percentile(normal_errors, 95)
print(f"Threshold: {threshold:.4f}")

preds_ae = (reconstruction_errors > threshold).astype(int)
print(f"Anomalies detected: {preds_ae.sum()}")

In [ ]:
print(classification_report(y, preds_ae, target_names=['Normal', 'Anomaly']))

cm = confusion_matrix(y, preds_ae)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Anomaly'])
disp.plot(cmap='Blues')
plt.title('Autoencoder (sklearn) — Confusion Matrix')
plt.show()

In [ ]:
joblib.dump(autoencoder, '../models/autoencoder.pkl')
np.save('../models/ae_threshold.npy', threshold)
print("Saved")